# Seismic events overlaid on a mine map

In [ ]:
import matplotlib.pyplot as plt

import geopandas as gpd
import numpy as np

In [ ]:
thickness_between = gpd.read_file('mocnost mezilozi lines.shp')

# thickness_between.plot()
# plt.title('Mocnost meziloží')
# plt.show()

In [ ]:
from seismi.dxf import load_dxf

mines_map = load_dxf('mines_map.dxf')

In [ ]:
from seismi.xls import read_seismic_events, read_measured_points, read_interpolated

seismic_events = read_seismic_events('SL jevy pri dobyvaní R 140 704.xlsx')
depths = read_measured_points('sloj 40 hloubka vrty.xlsx')
depths_interpolated = read_interpolated('sloj 40 hloubka grid.xlsx')

In [ ]:
from seismi.plots import plot_basic_map
fig, ax = plot_basic_map(mines_map, seismic_events)
plt.show()

Now this will be out of alignment:

In [ ]:
fig, ax = plot_basic_map(mines_map, seismic_events, thickness=thickness_between)
plt.show()

In [ ]:
from seismi.plots import create_interactive_map

m = create_interactive_map(
    seismic_events,
    depths=depths,
    depths_interpolated=depths_interpolated,
    sample_events=200,
    sample_interpolated=1000,
)
m

In [ ]:
from seismi.nn import prepare_data, BiggerNN, train_model, evaluate_model

(X_train, X_test, y_train, y_test,
 X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor,
 scaler) = prepare_data(seismic_events, depths_interpolated)

In [ ]:
model = BiggerNN()
model, losses = train_model(model, X_train_tensor, y_train_tensor, epochs=400)

In [ ]:
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Progress')
plt.show()

In [ ]:
preds, mse = evaluate_model(model, X_test_tensor, y_test)

In [ ]:
from seismi.plots import plot_model_results, create_interactive_prediction_map
figures = plot_model_results(y_test, preds, X_test)
for fig in figures:
    plt.figure(fig.number)
    plt.show()

In [ ]:
m = create_interactive_prediction_map(X_test, y_test, preds, scaler)
m